# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.16 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- **Lecture 01**: from "Text: Essential for Data Labels and Categories", used the `text.upper()` and `text.strip()` to make upper case and remove the space at front and end, in task 2.3, 3.1, 3.2
-**Lecture 02**: from "Dictionaries and Sets", built dictionary for value mapping(sex terms,visit terms, and etc) in task 3.1, 3.2, 4.1
- **Lecture 03**: from "NumPy Arrays", converted complete_pairs into an array with dtype=float (Task 4.3). Its shape is (27149, 2): 27149 rows, one per patient with both readings, and 2 columns, baseline then week12. I used axis=0 because it operates down each column across patients, giving one mean per visit with about 143.01 mmHg at baseline and 137.15 mmHg at week12.
- **Lecture 04**: from "Selecting with .loc and .iloc", using the .loc function to make the summary table in task 2.3,3.1,3.2
- **Lecture 05**: from "Reference Card: Finding missing values", use isna() in task 2.3,3.1,3.2 to find missing value in summary tables
- **Lecture 06**: from "Joining Tables with pd.merge()", use pd.merge in task 4.2 and 4.4 to merge tables.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%script "C:\Program Files\Git\bin\bash.exe"
# I am using Windows, so I need to run the bash command in a separate process.
# The %%script magic is a convenient way to do that.
wc -l data/*.csv | tee output/file_inventory.txt

      6 data/sites.csv
  26298 data/visits_north.csv
  31075 data/visits_south.csv
  57379 total


### 1.2 Profile each table with a function

In [3]:
manifest["files"]

{'patients.parquet': {'sha256': 'aa795b05777565cce6af68602e3fc7c286a2483f53f9a6d99a44da410c1214dd'},
 'sites.csv': {'sha256': 'daaa4f3fc475900db905590faf6524f34143e78b73633b1794de4c6cf4b56a77'},
 'visits_north.csv': {'sha256': 'f05014a1d281ee5e7732559d31fcf7a0e6e2b44f027b66e31229097484226a79'},
 'visits_south.csv': {'sha256': 'c19644241228ba7ef007d1aa7386516d80af5340b633401da67e9f59f3383214'},
 'a1c_wide.parquet': {'sha256': '934dc120b78d492666b7fbc7872d5fc4d0f20ab9a678bdb7559f00536a2cfa6a'}}

In [4]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.

def profile(name, table):
    """Return a dict with the four values in README Task 1.2 for the given table."""
    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": table.duplicated().sum()
    }
results = []
for name in manifest["files"]:
    tbl = raw[Path(name).stem]
    profile_data = profile(name, tbl)
    results.append(profile_data)
    print(profile_data)

pd.DataFrame(results).to_csv(PROFILE_PATH, index=False)

{'table': 'patients.parquet', 'rows': 30045, 'columns': 4, 'exact_duplicate_rows': np.int64(45)}
{'table': 'sites.csv', 'rows': 5, 'columns': 3, 'exact_duplicate_rows': np.int64(0)}
{'table': 'visits_north.csv', 'rows': 26297, 'columns': 6, 'exact_duplicate_rows': np.int64(35)}
{'table': 'visits_south.csv', 'rows': 31074, 'columns': 6, 'exact_duplicate_rows': np.int64(35)}
{'table': 'a1c_wide.parquet', 'rows': 29424, 'columns': 3, 'exact_duplicate_rows': np.int64(0)}


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

### Data contract (cleaned tables)

| Cleaned table | Grain: one row is | Key |
| --- | --- | --- |
| `patients_clean` | One registered patient | `patient_id` |
| `sites_current` | One site, current record only | `site_code` |
| `visits_clean`  | One clinic visit of one patient, keeping the later recheck | `patient_id` + `visit` (composite) |
| `a1c_wide`  | One patient's central lab results | `patient_id` |
| `a1c_long`  | One A1c result for one patient at one visit | `patient_id` + `visit` (composite) |

**Raw rows that violate these keys**

- **Exact repeats:** some raw rows are identical copies of an earlier row (counted in `exact_duplicate_rows` in `table_profile.csv`). In `patients.parquet` they repeat `patient_id`; cleaning removes them and keeps the first copy.
- **Visit rechecks:** after stacking, some `patient_id` + `visit` pairs appear more than once because the patient was measured again at the same visit. These are not exact copies (the time and readings differ), so the key is restored by keeping the later recheck. Visit labels must be normalized to `baseline`/`week12` first, or a spelling such as `wk12` would hide a repeat.
- **Current/retired site versions:** `sites.csv` holds more than one row for some `site_code`, one `current` and one or more `retired`. Keeping only `current` records gives one row per site.

Codes that break the rules but do not repeat a key, such as `site_code` spellings needing normalization or `E04` missing from `sites.csv`, are handled in cleaning and the joins rather than here.

**Expected cardinality of the Task 4.4 joins**

- `visits_clean` → `patients_clean` on `patient_id`: many-to-one. 
- `patients_clean` → `sites_current` on `site_code`: many-to-one. 
- `visits_clean` → `a1c_long` on `patient_id` + `visit`: one-to-one. 

### 2.2 Stack the two visit files

In [5]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
visits_raw = []
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
south["source_file"] = "visits_south.csv"

north = raw["visits_north"].copy(deep=True)
north["source_file"] = "visits_north.csv"

visits_raw = pd.concat([north, south], ignore_index=True)   
print(visits_raw.shape)

(57371, 7)


### 2.3 Audit the raw tables

In [6]:
visits_raw.columns

Index(['patient_id', 'visit', 'visit_date', 'visit_time', 'sbp_mmhg',
       'weight_kg', 'source_file'],
      dtype='str')

In [7]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.

visit_map = {
    "baseline": "baseline", "bl": "baseline",
    "week12": "week12", "wk12": "week12", "week 12": "week12",
}
visit_pairs = pd.DataFrame({
    "patient_id": visits_raw["patient_id"],
    "visit": visits_raw["visit"].str.strip().str.lower().map(visit_map),
})

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        raw["patients"].loc[(raw["patients"]["gender"].str.strip() != "") & 
                            (~raw["patients"]["gender"].isin(["X", "M", "F"]))].shape[0],
                            raw["patients"].loc[raw["patients"]["gender"].str.strip() == ""].shape[0],
                            raw["patients"].loc[raw["patients"]["birth_year"] == "9999"].shape[0],
                            raw["patients"].loc[raw["patients"]["site_code"] != raw["patients"]["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")].shape[0],
                            raw["sites"].duplicated(subset=["site_code"], keep=False).sum(),
                            visits_raw.loc[~visits_raw["visit"].isin(["baseline", "week12"])].shape[0],
                            visits_raw.loc[visits_raw["sbp_mmhg"] == "999"].shape[0],
                            visits_raw.loc[
                                (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") != 999)
                                & (
                                    (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") < 70)
                                    | (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") > 250)
                                )
                            ].shape[0],
                            (~visits_raw["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
                            | pd.to_datetime(visits_raw["visit_date"], format="%Y-%m-%d", errors="coerce").isna()).sum(),
                            visit_pairs.duplicated(subset=["patient_id", "visit"], keep=False).sum()
                        ]
        
    })
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
issue_audit.to_csv(AUDIT_PATH, index=False)
issue_audit

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [8]:
print(raw["patients"]["gender"].unique())
raw["patients"]["gender"].value_counts()

<ArrowStringArray>
[     'woman',          'F', 'Non-Binary',          'M',    'Female ',
     'female',       'male',         'm ',       'MALE',        'man',
       'Male',         ' F',     'FEMALE',          'f',          'X',
           '',         ' M',    'Unknown',    'unknown',          ' ',
  'nonbinary',          'x',         'NB', 'non-binary',      'Woman']
Length: 25, dtype: string


gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

In [9]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep="first")
gender_map = {
    "f": "F", 
    "female": "F", 
    "woman": "F",
    "m": "M", 
    "male": "M", 
    "man": "M",
    "x": "X", 
    "nonbinary": "X", 
    "non-binary": "X", 
    "nb": "X",
}
patients_clean["gender"] = (
    patients_clean["gender"].str.strip().str.lower().map(gender_map).astype("category")
)

birth_year = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean["birth_year"] = birth_year.mask(birth_year == 9999).astype("Int64")

patients_clean["site_code"] = (
    patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")
)

patients_clean = patients_clean.reset_index(drop=True)



In [10]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

patients_clean_summary = pd.DataFrame({
    "metric": ["patients",
               "birth_year missing",
               "birth_year min",
               "birth_year max",
               "site_code values",
               "patients at N01",
               "patients at S02",
               "patients at E04"],
    "value": [ patients_clean.shape[0],
               patients_clean["birth_year"].isna().sum(),
               patients_clean["birth_year"].min(),
               patients_clean["birth_year"].max(),
               patients_clean["site_code"].nunique(),
               patients_clean.loc[patients_clean["site_code"] == "N01"].shape[0],
               patients_clean.loc[patients_clean["site_code"] == "S02"].shape[0],
               patients_clean.loc[patients_clean["site_code"] == "E04"].shape[0]
    ]
})

patients_clean_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

patients_clean_summary

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


### 3.2 Clean the visits

In [11]:
visits_raw

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


In [12]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = visits_raw.copy(deep=True)

visits_clean.drop_duplicates(keep="first", inplace=True)

print(visits_clean["visit"].value_counts())

visit_map = {
    "baseline": "baseline", "bl": "baseline",
    "week12": "week12", "wk12": "week12", "week 12": "week12",
}
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")


exact = visits_clean["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
parsed = pd.to_datetime(visits_clean["visit_date"], format="%Y-%m-%d", errors="coerce")
visits_clean["visit_date"] = parsed.where(exact)



visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]


In [13]:
rechecks = visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)]
display(rechecks.sort_values(["patient_id", "visit", "visit_time"]))

visits_clean = (
    visits_clean.sort_values("visit_time", kind="stable")
    .drop_duplicates(subset=["patient_id", "visit"], keep="last")
)



,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [14]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).
visits_clean = (
    visits_clean.sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
    [["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
)
visits_clean_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",
    ],
    "value": [
        visits_clean.shape[0],
        visits_clean.loc[visits_clean["visit"] == "baseline"].shape[0],
        visits_clean.loc[visits_clean["visit"] == "week12"].shape[0],
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        visits_clean.loc[visits_clean["source_file"] == "visits_north.csv"].shape[0],
        visits_clean.loc[visits_clean["source_file"] == "visits_south.csv"].shape[0]
    ],
})
visits_clean_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_clean_summary

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [15]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table =decision_table = pd.DataFrame({
    "table": ["patients", "patients", "patients", "patients",
              "visits", "visits", "visits", "visits",
              "sites"],
    "issue": [
        "exact duplicate rows",
        "gender spellings that are not a code",
        "site codes needing normalization",
        "birth_year sentinel 9999",
        "weight column named wt_kg in visits_south.csv",
        "sbp sentinel 999 and readings outside 70 through 250",
        "impossible and loosely written visit dates",
        "repeated patient_id and visit pairs",
        "repeated site_code rows",
    ],
    "decision": [
        "I removed the duplication, keeping the first",  
        "I created a mapping to standardize the gender spellings to a consistent code.",
        "I standardized the site codes by removing hyphens and spaces, and converting them to uppercase.",
        "I set the sentinel value 9999 to NaN, as it represents missing data.",
        "I renamed the weight column to weight_kg for clarity.",
        "I set the sentinel value 999 to NaN and filtered out readings outside the range of 70 through 250.", 
        "I converted the visit dates to a standard format and set any impossible or loosely written dates to NaN.",  
        "I kept the later reading for repeated patient_id and visit pairs, as it is likely to be more up to date.",
        "I removed the duplicate site_code rows(by only keeping the current ones), as they are redundant.", 
    ],
    "reason": [
        "Informations are redundant, so only one instance is needed. And redundant information can lead to error in further analysis.", 
        "Different expression and empty values can lead to confusion during analysis. This ensures uniformity in the dataset and avoids confusion.", 
        "Site codes are in different formats, which can lead to confusion during analysis",
        "9999 represents missing data, keeping them would contaminate the max.", 
        "We should only keep the weight column with a clear name to avoid confusion during analysis.", 
        "There are dummy values for missing data and values exceeding the valid range.", 
        "Keeping the impossible and loosely written dates would contaminate the min and max, and lead to confusion during analysis.", 
        "There are repeated visits for the same patient in the save visit table, which can lead to confusion during analysis. Keeping the later reading is likely to be more up to date.", 
        "To ensure many to one relationship between site_code and site_name, and avoid confusion during analysis.",
    ],
})
decision_table.to_csv(DECISION_LOG_PATH, index=False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [16]:
raw["a1c_wide"]

,patient_id,a1c_baseline,a1c_week12
0,P00001,7.3,NaN
1,P00002,8.4,7.4
2,P00003,5.9,5.6
3,P00004,7.3,7.1
4,P00005,7.9,7.6
...,...,...,...
29419,P90026,6.7,6.1
29420,P90027,7.6,7.4
29421,P90028,7.3,7.0
29422,P90029,8.6,NaN


In [17]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).

a1c_long = pd.melt(raw["a1c_wide"], id_vars=["patient_id"], value_vars=["a1c_baseline", "a1c_week12"], var_name="visit", value_name="a1c_pct")
clean_dict = {
    "a1c_baseline": "baseline",
    "a1c_week12": "week12"
}
a1c_long["visit"] = a1c_long["visit"].map(clean_dict)

assert a1c_long["a1c_pct"].dtype == "float64", a1c_long["a1c_pct"].dtype
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

summary_a1c = pd.DataFrame({
    "visit":["baseline", "week12"],
    "results": [(a1c_long["visit"] == "baseline").sum(), (a1c_long["visit"] == "week12").sum()],
    "a1c_pct_mean": [round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2), round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2)],
})

summary_a1c.to_csv(A1C_SUMMARY_PATH, index=False)

### 4.2 Check the keys before joining

In [18]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]

try:
    patients_sites = pd.merge(patients_clean, sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print(e)
display(sites[sites.duplicated(subset=["site_code"], keep=False)])
sites_current = sites.loc[sites["record_status"] == "current", ["site_code", "site_name"]]
display(sites_current)


Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [19]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.
visits_a1c = pd.merge(visits_clean,a1c_long, on = ["patient_id", "visit"], how = "outer", indicator=True)
print(visits_a1c["_merge"].value_counts())

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule",
    ],
    "count": [
        (visits_a1c["_merge"] == "both").sum(),
        (visits_a1c["_merge"] == "left_only").sum(),
        (visits_a1c["_merge"] == "right_only").sum(),
        (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
        (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
        (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
        len(sites) - len(sites_current),
    ],
})

merge_audit.to_csv("output/merge_audit.csv", index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64


,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [20]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg")
sbp_wide = sbp_wide.reset_index()
sbp_wide.columns.name = None

sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)].to_csv(
    "output/sbp_wide_sample.csv", index=False
)
sbp_wide

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
1,P00002,132,129,-3
2,P00003,148,137,-11
3,P00004,142,<NA>,<NA>
4,P00005,139,127,-12
...,...,...,...,...
29594,P30036,127,126,-1
29595,P30037,135,127,-8
29596,P30038,135,135,0
29597,P30039,142,138,-4


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [21]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

complete_pairs = sbp_wide[["baseline", "week12"]].dropna()

sbp_array = np.array(complete_pairs, dtype=float)
print(sbp_array.shape)

visit_means = sbp_array.mean(axis=0)
print(visit_means)

(27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [22]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
n_visits = len(visits_clean)
analysis = pd.merge(
    visits_clean, patients_clean,
    on="patient_id", how="left", validate="many_to_one",
)
print("after patients:", len(analysis), n_visits)

analysis = pd.merge(
    analysis, sites_current,
    on="site_code", how="left", validate="many_to_one",
)
print("after sites:", len(analysis), n_visits)

analysis = pd.merge(
    analysis, a1c_long,
    on=["patient_id", "visit"], how="left", validate="one_to_one",
)
print("after a1c:", len(analysis), n_visits)

columns = [
    "patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct",
    "gender", "birth_year", "site_code", "site_name", "source_file",
]

analysis = analysis[columns].astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

analysis.dtypes


after patients: 57181 57181
after sites: 57181 57181
after a1c: 57181 57181


patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [23]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

sbp_present = analysis["sbp_mmhg"].dropna()
site_present = analysis["site_name"].dropna()

checks = pd.Series({
    "analysis has as many rows as visits_clean":
        len(analysis) == len(visits_clean),
    "no patient_id and visit pair repeats in analysis":
        not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "every sbp_mmhg that is present in analysis is from 70 through 250":
        bool(sbp_present.between(70, 250).all()),
    "every site_name that is present is a current site name":
        bool(site_present.isin(sites_current["site_name"]).all()),
    "sbp_wide['patient_id'] is unique":
        sbp_wide["patient_id"].is_unique,
    "every raw table still equals its snapshot":
        all(raw[name].equals(raw_snapshot[name]) for name in raw),
})
display(checks)
assert checks.all(), checks[~checks]

analysis has as many rows as visits_clean                            True
no patient_id and visit pair repeats in analysis                     True
every sbp_mmhg that is present in analysis is from 70 through 250    True
every site_name that is present is a current site name               True
sbp_wide['patient_id'] is unique                                     True
every raw table still equals its snapshot                            True
dtype: bool

### 5.2 Save the analysis extract

In [24]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

print(len(analysis_sample), visits_clean["patient_id"].isin(SAMPLE_IDS).sum())
analysis_sample

28 28


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [25]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.

round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
    },
    parse_dates=["visit_date"],
)

print(round_trip.equals(analysis_sample))

True
